In [13]:
import numpy as np
import pandas as pd
from numba import njit, prange
import multiprocessing as mp
from scipy.linalg import null_space
import time
from functools import partial
import itertools


In [2]:
@njit
def angle_between(v1: np.ndarray, v2: np.ndarray) -> float:
    norm_v1 = np.sqrt(np.dot(v1, v1))
    norm_v2 = np.sqrt(np.dot(v2, v2))
    
    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0  # Handle zero-length vectors gracefully
    
    cosine_angle = np.dot(v1, v2) / (norm_v1 * norm_v2)
    
    # Clamp the cosine value to avoid numerical errors
    if cosine_angle < -1.0:
        cosine_angle = -1.0
    elif cosine_angle > 1.0:
        cosine_angle = 1.0
    
    angle_rad = np.arccos(cosine_angle)
    return np.rad2deg(angle_rad) 

In [3]:
a1 = np.array([ 3.16599989, 0.        ])
a2 = np.array([-1.58300031,  2.74183612    ])

b1 = a1
b2 = a2


In [8]:
theta = np.deg2rad(27.80)

R = np.array(
    [[np.cos(theta), -np.sin(theta)], 
     [np.sin(theta), np.cos(theta)]],
    dtype=np.float64,
)

b1r = R @ b1
b2r = R @ b2

In [10]:
# Define the solution constraints
VAR_MIN, VAR_MAX = -5, 5
ERROR_TOLERANCE = 0.1 # New condition: ||Av|| < 0.1

# Construct the matrix A for the equation Av = 0
A = np.array([
    [a1[0], a2[0], -b1r[0], -b2r[0]],
    [a1[1], a2[1], -b1r[1], -b2r[1]]
])

# --- 2. Find Null Space (The Guide for Our Search) ---

K = null_space(A)
k1 = K[:, 0]
k2 = K[:, 1]
print("Null space basis (will guide our search):")
print(K)


Null space basis (will guide our search):
[[ 0.66981442 -0.40297925]
 [ 0.39251167  0.4846555 ]
 [ 0.62352718  0.01304675]
 [ 0.09217959  0.77623872]]


In [11]:
# --- 3. Determine Parameter Bounds for c1 and c2 ---
# This estimation is a heuristic, but sufficient for these ranges
c_bound_estimate = VAR_MAX / (np.abs(K).sum(axis=1).min() + 1e-9)
c1_range = np.array([-c_bound_estimate, c_bound_estimate])
c2_range = np.array([-c_bound_estimate, c_bound_estimate])
print(f"\nEstimated parameter search range: c1, c2 in ~[{c1_range[0]:.0f}, {c1_range[1]:.0f}]")




Estimated parameter search range: c1, c2 in ~[-8, 8]


In [23]:
# --- 4. Generate and Test Integer Candidates in the Neighborhood ---

print(f"\nSearching for integer vectors where the error is < {ERROR_TOLERANCE}...")
start_time = time.time()

found_solutions = set()

param_step = 0.5 # A reasonable step to sample the plane
c1_grid = np.arange(np.floor(c1_range[0]), np.ceil(c1_range[1]), param_step)
c2_grid = np.arange(np.floor(c2_range[0]), np.ceil(c2_range[1]), param_step)

# Generate the 16 offsets for the 4D hypercube corners
offsets = list(itertools.product([0, 1], repeat=4))

for c1 in c1_grid:
    for c2 in c2_grid:
        # 1. Get a point on the ideal, zero-error plane
        v_float = c1*k1 + c2*k2
        
        # 2. Get the base integer corner of the hypercube
        floor_v = np.floor(v_float).astype(int)

        # 3. Iterate through all 16 neighboring integer points
        for offset in offsets:
            v_int_candidate = floor_v + offset
            v_int_tuple = tuple(v_int_candidate)

            # 4. Check if we've already tested this integer candidate
            if v_int_tuple in found_solutions:
                continue

            # 5. Perform the two checks on this new candidate
            if np.all((v_int_candidate >= VAR_MIN) & (v_int_candidate <= VAR_MAX)):
                residual_vector = A @ v_int_candidate
                error_norm = np.linalg.norm(residual_vector)

                if error_norm < ERROR_TOLERANCE:
                    found_solutions.add((v_int_tuple,error_norm))


print(f"\nFound {len(found_solutions)} solutions in {time.time() - start_time:.4f} seconds.")

if len(found_solutions) > 0:
    non_trivial_solutions = {s for s in found_solutions if s != (0, 0, 0, 0)}
    print(f"Found {len(non_trivial_solutions)} non-trivial solutions.")
    
    # Sort the solutions for consistent output
    sorted_solutions = sorted(list(non_trivial_solutions))
    
    print("\nSolutions (n1, n2, m1, m2):")
    for sol in sorted_solutions:
        print(sol)


Searching for integer vectors where the error is < 0.1...

Found 9 solutions in 0.1156 seconds.
Found 9 non-trivial solutions.

Solutions (n1, n2, m1, m2):
((np.int64(-5), np.int64(2), np.int64(-2), np.int64(5)), np.float64(0.0014602509942237929))
((np.int64(-4), np.int64(-1), np.int64(-3), np.int64(1)), np.float64(0.0008426697626803293))
((np.int64(-3), np.int64(-4), np.int64(-4), np.int64(-3)), np.float64(0.0008414350479256465))
((np.int64(-1), np.int64(3), np.int64(1), np.int64(4)), np.float64(0.0008426628175416775))
((np.int64(0), np.int64(0), np.int64(0), np.int64(0)), np.float64(0.0))
((np.int64(1), np.int64(-3), np.int64(-1), np.int64(-4)), np.float64(0.0008426628175416775))
((np.int64(3), np.int64(4), np.int64(4), np.int64(3)), np.float64(0.0008414350479256465))
((np.int64(4), np.int64(1), np.int64(3), np.int64(-1)), np.float64(0.0008426697626803293))
((np.int64(5), np.int64(-2), np.int64(2), np.int64(-5)), np.float64(0.0014602509942237929))


In [24]:
sorted_solutions

[((np.int64(-5), np.int64(2), np.int64(-2), np.int64(5)),
  np.float64(0.0014602509942237929)),
 ((np.int64(-4), np.int64(-1), np.int64(-3), np.int64(1)),
  np.float64(0.0008426697626803293)),
 ((np.int64(-3), np.int64(-4), np.int64(-4), np.int64(-3)),
  np.float64(0.0008414350479256465)),
 ((np.int64(-1), np.int64(3), np.int64(1), np.int64(4)),
  np.float64(0.0008426628175416775)),
 ((np.int64(0), np.int64(0), np.int64(0), np.int64(0)), np.float64(0.0)),
 ((np.int64(1), np.int64(-3), np.int64(-1), np.int64(-4)),
  np.float64(0.0008426628175416775)),
 ((np.int64(3), np.int64(4), np.int64(4), np.int64(3)),
  np.float64(0.0008414350479256465)),
 ((np.int64(4), np.int64(1), np.int64(3), np.int64(-1)),
  np.float64(0.0008426697626803293)),
 ((np.int64(5), np.int64(-2), np.int64(2), np.int64(-5)),
  np.float64(0.0014602509942237929))]

In [37]:
for sol in sorted_solutions:
    n1, n2, vd = sol[0][0], sol[0][1], sol[1]
    for j in sorted_solutions:
        n1p, n2p = j[0][0], j[0][1]
        #print(n1,n2,n1p,n2p)
        A1 = n1 * a1 + n2 * a2
        A2 = n1p * a1 + n2p * a2
        angle_A1_A2 = angle_between(A1,A2)
        delta_angle = np.abs(angle_A1_A2 - 60)
        if delta_angle < 0.1:
            print(n1,n2,n1p,n2p,vd,delta_angle, A1, A2)

        

-4 -1 -3 -4 0.0008426697626803293 6.4522558744783964e-06 [-11.08099925  -2.74183612] [ -3.16599843 -10.96734448]
-4 -1 -1 3 0.0008426697626803293 6.783810782451383e-06 [-11.08099925  -2.74183612] [-7.91500082  8.22550836]
-3 -4 -4 -1 0.0008414350479256465 6.4522558744783964e-06 [ -3.16599843 -10.96734448] [-11.08099925  -2.74183612]
-3 -4 1 -3 0.0008414350479256465 3.315549363946957e-07 [ -3.16599843 -10.96734448] [ 7.91500082 -8.22550836]
-1 3 -4 -1 0.0008426628175416775 6.783810782451383e-06 [-7.91500082  8.22550836] [-11.08099925  -2.74183612]
-1 3 3 4 0.0008426628175416775 3.315549363946957e-07 [-7.91500082  8.22550836] [ 3.16599843 10.96734448]
1 -3 -3 -4 0.0008426628175416775 3.315549363946957e-07 [ 7.91500082 -8.22550836] [ -3.16599843 -10.96734448]
1 -3 4 1 0.0008426628175416775 6.783810782451383e-06 [ 7.91500082 -8.22550836] [11.08099925  2.74183612]
3 4 -1 3 0.0008414350479256465 3.315549363946957e-07 [ 3.16599843 10.96734448] [-7.91500082  8.22550836]
3 4 4 1 0.0008414350479

In [34]:
for sol in sorted_solutions:
    print(sol)
    break

((np.int64(-5), np.int64(2), np.int64(-2), np.int64(5)), np.float64(0.0014602509942237929))


In [36]:
sol[1]

np.float64(0.0014602509942237929)